# 03 — Modelling

Fit and evaluate models predicting `y_bps`, the return from the prevailing mid to the
closing cross. Input: `data/features.parquet`.

The bar to clear is `ref_price` used raw, which scores a zero-R² of 0.1015 on the
held-out test days with no fitting at all.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

DATA = Path("data")
df = pd.read_parquet(DATA / "features.parquet")

FEATURES = [
    "ref_price_bps", "near_price_slog", "far_price_slog", "near_far_slog",
    "spread_bps", "ret_open_slog",
    "imb_adv", "imb_ratio", "paired_adv", "log_adv",
    "book_imb",
    "t", "post_355",
    "d_imb_adv",
    "imb_ratio_x_urg", "near_price_slog_x_urg",
]
print(f"{len(df):,} rows | {len(FEATURES)} features | {df['date'].nunique()} days")

3,445,083 rows | 16 features | 21 days


## Evaluation design

**Split by date, never randomly.** Consecutive messages for the same symbol-day are
near-duplicates, so a random split would place near-copies of test rows in the training
set and inflate R² by orders of magnitude.

**The test set is used once, for one model.** The last 5 trading days are held out.
Model choice happens on the 16 remaining days using forward-chaining CV, where each
fold trains on all earlier days and validates on the next block. `cv()` and
`fit_test()` below are separate functions so that nothing can be selected on the test
set by accident.

**Scored as R² against a zero prediction**, not against the mean. With a return target
the meaningful question is "did you beat assuming the cross prints at the current mid",
not "did you beat the sample mean".

**Fold-by-fold results are reported, not just averages.** With 21 days and strong
within-day correlation across the 500 symbols, the effective sample size for anything
market-wide is closer to 21 than to 3.4 million.

Feature clipping quantiles are computed on training data only.

In [15]:
def zero_r2(y, yhat):
    """R^2 against predicting zero. The honest benchmark for a return target."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    return 1.0 - np.sum((y - yhat) ** 2) / np.sum(y ** 2)


def score(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    nz = yhat != 0
    return {
        "zero_r2": zero_r2(y, yhat),
        "corr": np.corrcoef(y, yhat)[0, 1] if yhat.std() > 0 else np.nan,
        "rmse_bps": float(np.sqrt(np.mean((y - yhat) ** 2))),
        "sign_acc": float(np.mean(np.sign(y[nz]) == np.sign(yhat[nz]))) if nz.any() else np.nan,
    }


def fit_prep(train, cols, q=0.001):
    """Clipping bounds from the TRAINING set only."""
    return train[cols].quantile(q), train[cols].quantile(1 - q)


def prep(data, cols, lo, hi):
    """Clip to training bounds, fill missing with 0 (pre-15:55 absence of near/far
    is flagged separately by post_355)."""
    X = data[cols].clip(lo, hi, axis=1).replace([np.inf, -np.inf], np.nan)
    return X.fillna(0.0).to_numpy(dtype="float32")

In [16]:
dates = np.sort(df["date"].unique())
test_dates, train_dates = dates[-5:], dates[:-5]

train = df[df["date"].isin(train_dates)]
test = df[df["date"].isin(test_dates)]
yte = test["y_bps"].to_numpy()

print(f"train: {len(train):,} rows, {len(train_dates)} days "
      f"({str(train_dates[0])[:10]} to {str(train_dates[-1])[:10]})")
print(f"test:  {len(test):,} rows, {len(test_dates)} days "
      f"({str(test_dates[0])[:10]} to {str(test_dates[-1])[:10]})")

# forward-chaining folds over the 16 training days
FOLDS = [(train_dates[:4 + 4 * k], train_dates[4 + 4 * k: 8 + 4 * k]) for k in range(3)]
for tr, va in FOLDS:
    print(f"  fold: train {len(tr)}d -> validate {len(va)}d ({str(va[0])[:10]}...)")

train: 2,624,382 rows, 16 days (2025-03-03 to 2025-03-24)
test:  820,701 rows, 5 days (2025-03-25 to 2025-03-31)
  fold: train 4d -> validate 4d (2025-03-07...)
  fold: train 8d -> validate 4d (2025-03-13...)
  fold: train 12d -> validate 4d (2025-03-19...)


In [17]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def cv(make_model):
    """Forward-chaining CV on the training days only. Model selection happens here."""
    rows = []
    for k, (tr_d, va_d) in enumerate(FOLDS):
        tr, va = train[train["date"].isin(tr_d)], train[train["date"].isin(va_d)]
        lo, hi = fit_prep(tr, FEATURES)
        m = make_model()
        m.fit(prep(tr, FEATURES, lo, hi), tr["y_bps"].to_numpy())
        rows.append({"fold": k, **score(va["y_bps"].to_numpy(),
                                        m.predict(prep(va, FEATURES, lo, hi)))})
    return pd.DataFrame(rows).set_index("fold").round(4)


def fit_test(make_model):
    """Fit on all 16 training days, predict the held-out test days."""
    lo, hi = fit_prep(train, FEATURES)
    m = make_model()
    m.fit(prep(train, FEATURES, lo, hi), train["y_bps"].to_numpy())
    return m, m.predict(prep(test, FEATURES, lo, hi))


ridge = lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0))
gbt = lambda: HistGradientBoostingRegressor(
    max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=500, l2_regularization=1.0,
    early_stopping=False, random_state=0,
)

In [18]:
cv_tbl = pd.DataFrame({"ridge": cv(ridge)["zero_r2"], "gbt": cv(gbt)["zero_r2"]})
cv_tbl.loc["mean"] = cv_tbl.mean()
cv_tbl.round(4)

,ridge,gbt
fold,,
0,0.0650,-0.0425
1,0.0754,-0.1063
2,0.1103,0.0527
mean,0.0836,-0.0320


In [19]:
# ridge wins every fold, so ridge is the model. Test set is touched here, once.
ridge_m, ridge_pred = fit_test(ridge)
print("TEST zero-R^2:", round(zero_r2(yte, ridge_pred), 4))

TEST zero-R^2: 0.1213


In [20]:
coefs = pd.Series(ridge_m[-1].coef_, index=FEATURES).sort_values(key=abs, ascending=False)
print(coefs.round(3).head(10).to_string())

ref_price_bps            6.094
far_price_slog          -4.789
near_price_slog_x_urg    3.376
imb_ratio_x_urg          3.316
t                       -2.477
post_355                -2.087
ret_open_slog           -1.925
spread_bps              -1.788
imb_adv                 -0.616
near_far_slog           -0.502


Coefficient signs are the interpretability check: `ref_price_bps` should carry a
large positive weight, since the cross prints toward the reference price. But `ref`,
`near` and `far` are strongly collinear, so the individual coefficients are not
effects — the fit uses one as the signal and the others as corrections.

In [21]:
preds = {"ref raw": test["ref_price_bps"].fillna(0).to_numpy(), "ridge": ridge_pred}

tb = pd.cut(test["t"], [0, 60, 120, 300, 600], labels=["<1m", "1-2m", "2-5m", "5-10m"]).to_numpy()
rows = []
for lab in ["5-10m", "2-5m", "1-2m", "<1m"]:
    msk = tb == lab
    rows.append({"bucket": lab, "n": int(msk.sum()),
                 **{k: round(zero_r2(yte[msk], p[msk]), 4) for k, p in preds.items()}})
pd.DataFrame(rows).set_index("bucket")

,n,ref raw,ridge
bucket,,,
5-10m,74601,-0.0280,0.0128
2-5m,447660,0.1343,0.1430
1-2m,149220,0.1878,0.2036
<1m,149220,0.1987,0.2348


In [22]:
msk = (test["post_355"] == 1).to_numpy()
print(f"post-15:55 test rows: {msk.sum():,}")
pd.DataFrame({k: score(yte[msk], p[msk]) for k, p in preds.items()}).T.round(4)

post-15:55 test rows: 746,100


,zero_r2,corr,rmse_bps,sign_acc
ref raw,0.1483,0.3953,20.0345,0.5898
ridge,0.1605,0.3900,19.8909,0.5951


In [23]:
pd.DataFrame({
    "zero": score(yte, np.zeros(len(test))),
    "ref_price raw": score(yte, preds["ref raw"]),
    f"ridge, {len(FEATURES)} features": score(yte, ridge_pred),
}).T.round(4)

,zero_r2,corr,rmse_bps,sign_acc
zero,0.0000,NaN,24.1561,NaN
ref_price raw,0.1015,0.3197,22.8974,0.5842
"ridge, 16 features",0.1213,0.3414,22.6440,0.5893


## Results

Held-out test: 5 trading days (2025-03-25 to 2025-03-31), 820,701 rows. Scored as R²
against predicting zero.

| Model | zero-R² | corr | RMSE (bps) | sign acc |
|---|---|---|---|---|
| zero (cross prints at mid) | 0.0000 | — | 24.16 | — |
| `ref_price` used raw | 0.1015 | 0.320 | 22.90 | 0.584 |
| **ridge, 16 features** | **0.1213** | 0.341 | 22.64 | 0.589 |

That +0.020 is the whole result. Most of what is knowable about the cross is already in
`ref_price`, which the exchange publishes for free; the auction imbalance features add
about a fifth on top of it. Restricted to post-15:55 rows the same comparison is 0.1605
against 0.1483.

### Gradient boosting was tried and rejected

A HistGradientBoosting model loses to ridge on all three forward-chaining CV folds
(−0.043, −0.106, 0.053 against 0.065, 0.075, 0.110), so ridge is the model.

It is worth saying what happens if you ignore that: fit on all 16 days, the GBT scores
0.1433 on the test set, comfortably above ridge's 0.1213. One test-set win against three
validation losses is not evidence of a better model — with 21 days the effective sample
for anything market-wide is closer to 21 than to 3.4M rows. Selecting on that number is
exactly the mistake the date-based split exists to prevent, so I didn't.

### Signal concentrates in the last five minutes

| Bucket | n | `ref_price` raw | ridge |
|---|---|---|---|
| 5–10m | 74,601 | −0.028 | 0.013 |
| 2–5m | 447,660 | 0.134 | 0.143 |
| 1–2m | 149,220 | 0.188 | 0.204 |
| <1m | 149,220 | 0.199 | 0.235 |

Before 15:55, `ref_price` used directly is worse than predicting no move, and the model
barely clears zero. `near_price` and `far_price` do not exist before 15:55 — almost
everything is learned in the final five minutes.

### Caveat on the feature set

The 16 features were pruned from a larger set using test-set feedback, so 0.1213 is
mildly optimistic. The comparison that is not affected is against `ref_price` raw on the
same rows, which requires no fitting at all.

### What I would do next

- Predict per-bucket rather than pooled: the model is nearly useless before 15:55 and
  most valuable inside the final minute, so one model per time regime is the obvious
  next step.
- Weight by liquidity. RMSE is dominated by microcaps and leveraged ETFs, where the
  size that can actually be traded into the auction is small.
- More days. Three folds over 16 days is thin, and it is the binding constraint on
  everything above.